# PADC / CIFAR — TWC 2023 Fig. 7(a)·Fig. 5 정밀 재검증

**이 노트북은 재학습하지 않습니다.** 이미 학습된 DeepJSCC-V (400 epochs)와 OraNet (150 epochs)의 **CIFAR10_validation / CIFAR100_test CSV**를 읽어, 논문과 동일한 CIFAR 실험 축에서 평가하고 목표 PSNR의 실현 가능성을 구분합니다.

- **Fig. 7(a):** CIFAR100 10,000장, AWGN, SNR `0,3,...,24 dB`, R=`1/16,1/12,1/6`. 각 이미지 오차 제곱을 평균하여 **prediction MSE (dB²)** 산출.
- **Fig. 5-style:** 8개 CR 후보, 목표 `24,26,28,30,32,34 dB`. **Data-level 단일 CR**의 CIFAR100 집합 평균 기준(논문에 가까운 사후 벤치마크) 및 CIFAR10 validation lookup(배포 가능한 교차 데이터셋 기준)을 구분.
- **Instance-level PADC:** OraNet의 이미지별 CR 선택 vs 실제 PSNR 기반 Genie (사후 benchmark). **Feasible-only 성공률** 및 성공한 사례의 추가 전송률 계산.
- **Calibrated sensitivity:** CIFAR10 validation에서 얻은 경험적 95% 과대예측 마진을 적용한 대조 실험 (통계적 보장을 주장하지 않음).

**중요한 한계:** 공식 GitHub의 ADJSCC 4개 독립 모델을 학습하지 않았다면 Fig. 5의 **ADJSCC 비교까지 재현했다고 주장할 수 없습니다**. 논문은 [Zhang et al., IEEE TWC 2023](https://doi.org/10.1109/TWC.2023.3234408), 모델은 [원저자 GitHub](https://github.com/wyzhang-ustb/Predictive-and-Adaptive-Deep-Coding-for-Wireless-Image-Transmission-in-Semantic-Communication)입니다.


## 0. 실행 환경

Google Colab **CPU 또는 GPU 모두 가능**. 기존 평가 CSV가 Google Drive에 있어야 합니다. 원래 노트북의 학습 셀을 다시 실행할 필요가 없습니다. **한 번에 전체 실행**해도 됩니다.


In [ ]:
# ===== EDIT ME =====
USE_GOOGLE_DRIVE = True
EXPERIMENT_DIR = '/content/drive/MyDrive/PADC_reproduction_CIFAR_fresh_20261022_v1/paper/awgn'
CIFAR100_ROOT = '/content/drive/MyDrive/datasets'  # optional checkpoint prediction spot-check

EXPECTED_IMAGES = 10000
RUN_CHECKPOINT_SPOTCHECK = True  # recommended: verify that cached OraNet predictions match current checkpoint
SPOTCHECK_TOLERANCE_DB = 0.025  # allow small floating-point/device differences
SEED = 2026

print('Experiment:',EXPERIMENT_DIR)
print('No training; only reproducibility checks and post-hoc analysis.')

In [ ]:
import os, sys, json, hashlib, warnings, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('This configuration expects Google Colab. Set USE_GOOGLE_DRIVE=False for local testing.') from exc
    drive.mount('/content/drive',force_remount=False)

EXPERIMENT_DIR = Path(EXPERIMENT_DIR)
RESULT_DIR = EXPERIMENT_DIR / 'results'
OUT = RESULT_DIR / 'cifar_paper_fig7_fig5_reaudit'
OUT.mkdir(parents=True,exist_ok=True)
CIFAR_VAL_CSV = RESULT_DIR / 'cifar10_val_grid.csv'
CIFAR_TEST_CSV = RESULT_DIR / 'cifar100_test_grid.csv'
JSCC_BEST = EXPERIMENT_DIR/'checkpoints'/'jscc_v_awgn_cifar10_best.pt'
ORACLE_BEST = EXPERIMENT_DIR/'checkpoints'/'oracle_awgn_cifar10_best.pt'
print('Result output:',OUT)
for p in [CIFAR_VAL_CSV,CIFAR_TEST_CSV]:
    if not p.is_file(): raise FileNotFoundError(f'{p} not found. Run the CIFAR evaluation cell from your original notebook.')

## 1. 정밀 재평가 함수

아래 구현은 원본 학습 코드를 수정하거나 네트워크를 재학습하지 않습니다. GPU 없이 CSV만으로 결정적인 통계량을 계산합니다. 특정 SNR·CR 관측치가 없는 경우 **보간하지 않고 오류를 발생**시킵니다.

In [ ]:
"""Post-hoc PADC CIFAR evaluation audit (no model fitting; CPU-only).

This module is intentionally independent of the author's network runtime: it
only reads evaluation CSVs produced with the user's frozen checkpoints.
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import hashlib
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG7_SNR = np.arange(0, 25, 3, dtype=float)
FIG7_CHANNELS = [6, 8, 16]  # physically R=1/16, 1/12, 1/6 in 48-channel architecture
FIG5_THRESHOLDS = np.array([24., 26., 28., 30., 32., 34.])
FIG5_CHANNELS = [6, 7, 8, 10, 12, 16, 24, 32]  # R in {1/16,1/14,1/12,1/10,1/8,1/6,1/4,1/3}

REQUIRED = ['image_id','snr_db','requested_r','active_channels','effective_r','psnr_db','predicted_psnr_db']


def file_sha256(path, chunk=4*1024*1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda: f.read(chunk), b''):
            h.update(part)
    return h.hexdigest()


def read_grid(path, expected_dataset, expected_n=10000, expected_channel='awgn'):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run the evaluation cell in the original notebook before this audit.')
    df = pd.read_csv(path, usecols=lambda s: s in REQUIRED or s in ('dataset','channel','channel_repeats'))
    absent = sorted(set(REQUIRED)-set(df.columns))
    if absent:
        raise ValueError(f'{path.name}: missing essential CSV columns: {absent}')
    df['image_id'] = df['image_id'].astype(str)
    if 'dataset' in df.columns and not df['dataset'].eq(expected_dataset).all():
        raise ValueError(f'Expected dataset {expected_dataset}, got {df.dataset.unique()}')
    if 'channel' in df.columns and not df['channel'].eq(expected_channel).all():
        raise ValueError(f'Expected channel {expected_channel}, got {df.channel.unique()}')
    nimages = df.image_id.nunique()
    if expected_n is not None and nimages != expected_n:
        raise ValueError(f'{path.name}: expected {expected_n} unique images, found {nimages}. CSV may be partial.')
    if not np.isfinite(df[['snr_db','effective_r','psnr_db','predicted_psnr_db']].to_numpy()).all():
        raise ValueError(f'{path.name}: non-finite grid values')
    if df.duplicated(['image_id','snr_db','active_channels']).any():
        raise ValueError(f'{path.name}: duplicate image/SNR/active-channel rows')
    if not np.allclose(df.effective_r, df.active_channels/96.,rtol=0,atol=1e-6):
        raise ValueError(f'{path.name}: effective rate must be active_channels/96')
    if 'channel_repeats' in df.columns:
        u = df.channel_repeats.unique()
        if len(u) != 1:
            raise ValueError('channel_repeats differs across rows')
        print(f'{path.name}: stochastic-channel repeats per image/SNR/rate = {u[0]}')
    meta = path.with_suffix('.meta.json')
    md = json.loads(meta.read_text()) if meta.exists() else None
    if md is None:
        warnings.warn(f'{path.name}: missing .meta.json provenance')
    else:
        if md.get('dataset_name')!=expected_dataset or md.get('channel')!=expected_channel:
            raise ValueError(f'{path.name}: metadata dataset or channel mismatch')
        # Old notebook cache signatures record JSCC SHA, but NOT OraNet SHA.
        if 'oracle_sha256' not in md:
            warnings.warn(f'{path.name}: OraNet checkpoint SHA is missing from legacy cache. '
                          'The CSV predictions cannot be cryptographically tied to the present OraNet checkpoint.')
    print(f'{path.name}: {len(df):,} records, {nimages:,} images, '
          f'{df.snr_db.nunique()} SNR values, {df.active_channels.nunique()} rates')
    return df, md


@dataclass
class GridCube:
    image_ids: list
    snrs: np.ndarray
    channels: np.ndarray
    rates: np.ndarray
    actual: np.ndarray
    predicted: np.ndarray


def to_cube(df, snrs=FIG7_SNR, channels=FIG5_CHANNELS):
    snrs = np.asarray(snrs, float)
    channels = np.asarray(channels, int)
    d = df[df.snr_db.isin(snrs) & df.active_channels.isin(channels)].copy()
    images = sorted(d.image_id.unique())
    expected = len(images)*len(snrs)*len(channels)
    if len(d) != expected:
        raise ValueError(f'Incomplete rectangular SNR x rate grid: got {len(d):,}, expected {expected:,}.')
    d = d.set_index(['image_id','snr_db','active_channels']).sort_index()
    key = pd.MultiIndex.from_product([images,snrs,channels], names=['image_id','snr_db','active_channels'])
    aligned = d.reindex(key)
    if aligned[['psnr_db','predicted_psnr_db']].isna().any().any():
        raise ValueError('Incomplete grid after exact alignment. Do not interpolate or invent observations.')
    n = (len(images),len(snrs),len(channels))
    actual = aligned['psnr_db'].to_numpy(dtype=np.float64).reshape(n)
    predicted = aligned['predicted_psnr_db'].to_numpy(dtype=np.float64).reshape(n)
    return GridCube(images,snrs,channels,channels.astype(float)/96.,actual,predicted)


def fig7_metrics(test_df):
    d = test_df[test_df.snr_db.isin(FIG7_SNR) & test_df.active_channels.isin(FIG7_CHANNELS)].copy()
    if d.empty: raise ValueError('Fig7 SNR and CR grid is empty')
    d['err_db'] = d.predicted_psnr_db-d.psnr_db
    d['sq_error'] = d.err_db**2
    d['abs_error'] = d.err_db.abs()
    result = (d.groupby(['active_channels','snr_db'],as_index=False)
                .agg(n=('sq_error','size'),mse_db2=('sq_error','mean'),
                     mse_std_db2=('sq_error','std'),mae_db=('abs_error','mean'),
                     signed_bias_db=('err_db','mean')))
    result['physical_r']=result.active_channels/96.
    result['mse_se_db2']=result.mse_std_db2/np.sqrt(result.n)
    result['mse_ci95_low']=np.maximum(0,result.mse_db2-1.96*result.mse_se_db2)
    result['mse_ci95_high']=result.mse_db2+1.96*result.mse_se_db2
    result=result.sort_values(['active_channels','snr_db']).reset_index(drop=True)
    if len(result)!=3*9: raise AssertionError('Expected Fig7 27 rate/SNR combinations')
    return result


def val_margin(val_df, probs=(0.90,0.95)):
    err=val_df.predicted_psnr_db.to_numpy()-val_df.psnr_db.to_numpy()
    # This dataset participated in the checkpoint's validation/model selection;
    # these quantiles are descriptive, NOT distribution-free guarantees.
    return {f'overprediction_quantile_{int(q*100)}':max(0.,float(np.quantile(err,q))) for q in probs}


def choose_first_feasible(quality, thresholds):
    """Select minimum *available* effective rate; retain feasibility flag."""
    quality=np.asarray(quality, float)
    target=np.asarray(thresholds,float)
    # quality: (..., R); thresholds: (..., T) with same leading dims.
    hit = quality[...,None,:] >= target[..., :,None]
    can = hit.any(axis=-1)
    ix = np.where(can,hit.argmax(axis=-1),quality.shape[-1]-1)
    return ix,can


def data_level_summary(val_cube, test_cube, thresholds=FIG5_THRESHOLDS):
    if not np.allclose(val_cube.rates,test_cube.rates) or not np.allclose(val_cube.snrs,test_cube.snrs):
        raise ValueError('Validation/test rate and SNR axes must match')
    val_mean=val_cube.actual.mean(axis=0)
    test_mean=test_cube.actual.mean(axis=0)
    rows=[]
    for s,snr in enumerate(test_cube.snrs):
        for threshold in thresholds:
            val_ok=val_mean[s]>=threshold
            paper_ok=test_mean[s]>=threshold
            vind=int(np.flatnonzero(val_ok)[0]) if val_ok.any() else len(test_cube.rates)-1
            pind=int(np.flatnonzero(paper_ok)[0]) if paper_ok.any() else len(test_cube.rates)-1
            rows.append(dict(snr_db=float(snr),target_psnr_db=float(threshold),
                             data_cifar10_rate=float(test_cube.rates[vind]),
                             data_cifar10_feasible=bool(val_ok.any()),
                             data_cifar10_psnr_test=float(test_mean[s,vind]),
                             data_cifar10_satisfies_mean_test=bool(test_mean[s,vind]>=threshold),
                             paper_posthoc_cifar100_rate=float(test_cube.rates[pind]),
                             paper_posthoc_cifar100_feasible=bool(paper_ok.any()),
                             paper_posthoc_cifar100_psnr_test=float(test_mean[s,pind]),
                             paper_posthoc_cifar100_satisfies_mean_test=bool(test_mean[s,pind]>=threshold)))
    return pd.DataFrame(rows)


def instance_level_summary(cube, thresholds=FIG5_THRESHOLDS, margin=0.0, tag='raw'):
    """Vectorized per-image exhaustive rate optimizer and honest feasible-only metrics."""
    truth=cube.actual
    pred=cube.predicted - max(0.,float(margin))
    N,S,R=truth.shape
    tgt=np.asarray(thresholds,float)
    T=len(tgt)
    true_ok=(truth[:,:,None,:]>=tgt[None,None,:,None])
    pred_ok=(pred[:,:,None,:]>=tgt[None,None,:,None])
    feasible=true_ok.any(axis=3)
    pred_feasible=pred_ok.any(axis=3)
    gi=np.where(feasible,np.argmax(true_ok,axis=3),R-1)
    pi=np.where(pred_feasible,np.argmax(pred_ok,axis=3),R-1)
    truth_by_t=np.broadcast_to(truth[:,:,None,:],(N,S,T,R))
    tr_chosen=np.take_along_axis(truth_by_t,pi[...,None],axis=3)[...,0]
    satisfied=tr_chosen>=tgt[None,None,:]
    padc_r=cube.rates[pi]
    genie_r=cube.rates[gi]
    mask=(feasible & satisfied)
    rows=[]
    for s,snr in enumerate(cube.snrs):
        for t,threshold in enumerate(tgt):
            nf=int(feasible[:,s,t].sum());success=int((satisfied[:,s,t] & feasible[:,s,t]).sum())
            both=mask[:,s,t]
            wrong_low=(feasible[:,s,t]&(~satisfied[:,s,t]))
            regret=padc_r[both,s,t]-genie_r[both,s,t]
            rows.append(dict(variant=tag,margin_db=float(margin),
                snr_db=float(snr),target_psnr_db=float(threshold),n_images=N,
                n_feasible=nf,feasible_fraction=nf/N,
                success_all_fraction=float(satisfied[:,s,t].mean()),
                success_given_feasible=float(success/nf) if nf else np.nan,
                n_feasible_success=success,
                predicted_feasible_fraction=float(pred_feasible[:,s,t].mean()),
                missed_target_feasible_count=int(wrong_low.sum()),
                selected_below_genie_rate_count=int((padc_r[feasible[:,s,t],s,t]<genie_r[feasible[:,s,t],s,t]-1e-9).sum()),
                mean_padc_rate_all=float(padc_r[:,s,t].mean()),
                mean_genie_rate_all=float(genie_r[:,s,t].mean()),
                mean_padc_rate_feasible=float(padc_r[feasible[:,s,t],s,t].mean()) if nf else np.nan,
                mean_genie_rate_feasible=float(genie_r[feasible[:,s,t],s,t].mean()) if nf else np.nan,
                mean_excess_rate_if_success=float(regret.mean()) if len(regret) else np.nan,
                avg_selected_actual_psnr_db=float(tr_chosen[:,s,t].mean()),
                avg_selected_actual_psnr_feasible=float(tr_chosen[feasible[:,s,t],s,t].mean()) if nf else np.nan))
    result=pd.DataFrame(rows)
    detail=None  # don't emit 540k per-image cases by default, keep memory and Drive compact
    return result


def monotonicity_summary(cube):
    delta_true=np.diff(cube.actual,axis=-1)
    delta_pred=np.diff(cube.predicted,axis=-1)
    rows=[]
    for i,snr in enumerate(cube.snrs):
        rows.append(dict(snr_db=float(snr),
            actual_decreasing_pair_fraction=float((delta_true[:,i,:]<-1e-6).mean()),
            oranet_decreasing_pair_fraction=float((delta_pred[:,i,:]<-1e-6).mean()),
            images_with_actual_violation_fraction=float((delta_true[:,i,:]<-1e-6).any(axis=1).mean()),
            images_with_pred_violation_fraction=float((delta_pred[:,i,:]<-1e-6).any(axis=1).mean())))
    return pd.DataFrame(rows)


def aggregate_instance(summary):
    """Weighted global rate/success summary; avoid averaging per-SNR percentages when no feasible cases."""
    records=[]
    for tag,g in summary.groupby('variant'):
        nf=int(g.n_feasible.sum())
        total=int(g.n_images.sum())
        records.append(dict(variant=tag,n_total=total,n_feasible=nf,
            feasible_share=nf/total,
            success_given_feasible=(g.n_feasible_success.sum()/nf) if nf else np.nan,
            success_all_fraction=float(np.average(g.success_all_fraction,weights=g.n_images)),
            mean_padc_rate=float(np.average(g.mean_padc_rate_all,weights=g.n_images)),
            mean_genie_rate=float(np.average(g.mean_genie_rate_all,weights=g.n_images)),
            mean_excess_rate_if_success=(np.average(g.mean_excess_rate_if_success.dropna(),weights=(g.n_feasible_success.loc[g.mean_excess_rate_if_success.notna()])) if g.mean_excess_rate_if_success.notna().any() and g.n_feasible_success.loc[g.mean_excess_rate_if_success.notna()].sum()>0 else np.nan)))
    return pd.DataFrame(records)


def fig7_plot(fig7, dst):
    fig,ax=plt.subplots(figsize=(7.6,4.8))
    for active,d in fig7.groupby('active_channels'):
        r=active/96.
        label={6:'1/16',8:'1/12',16:'1/6'}[int(active)]
        ax.plot(d.snr_db,d.mse_db2,'o-',lw=1.6,label=f'R = {label} ({r:.4f})')
        ax.fill_between(d.snr_db.to_numpy(),d.mse_ci95_low.to_numpy(),d.mse_ci95_high.to_numpy(),alpha=.12)
    ax.set(xlim=(-.5,24.5),xlabel='Channel SNR (dB)',ylabel='PSNR prediction MSE (dB$^2$)',
           title='CIFAR100 / AWGN — paper Fig. 7(a) conditions')
    ax.set_xticks(FIG7_SNR)
    ax.grid(alpha=.3);ax.legend(loc='upper right',fontsize=9)
    fig.tight_layout()
    fig.savefig(dst,dpi=180);plt.show();plt.close(fig)


def fig5_plot(data_level,dst):
    fig,ax=plt.subplots(figsize=(8.2,5))
    for t,d in data_level.groupby('target_psnr_db'):
        p=ax.plot(d.snr_db,d.paper_posthoc_cifar100_rate,'o-',lw=1.5,label=f'{t:g} dB test-set mean')[0]
        ax.plot(d.snr_db,d.data_cifar10_rate,'--',lw=1,alpha=.7,color=p.get_color())
    ax.set(xlabel='Channel SNR (dB)',ylabel='Data-level selected effective CR (K/N)',
           title='CIFAR100 / AWGN — Fig. 5-style DeepJSCC-V data-level rates')
    ax.set_xticks(FIG7_SNR)
    ax.grid(alpha=.3);ax.legend(fontsize=8,ncol=2)
    fig.text(.5,-.01,'Solid: paper-style post-hoc CIFAR100 mean; dashed: deployment CIFAR10 validation lookup. No ADJSCC baseline.',ha='center',fontsize=8)
    fig.tight_layout()
    fig.savefig(dst,dpi=180,bbox_inches='tight');plt.show();plt.close(fig)


def instance_plot(summary,dst,metric='success_given_feasible',title=''):
    fig,ax=plt.subplots(figsize=(8.2,4.8))
    raw=summary[summary.variant=='raw']
    cal=summary[summary.variant!='raw']
    for t,d in raw.groupby('target_psnr_db'):
        p=ax.plot(d.snr_db,d[metric],'o-',lw=1.5,label=f'Raw, {t:g} dB')[0]
        c=cal[cal.target_psnr_db==t]
        if not c.empty:
            ax.plot(c.snr_db,c[metric],'--',lw=1,alpha=.85,color=p.get_color())
    ax.set(xlabel='Channel SNR (dB)',ylabel=metric.replace('_',' '),title=title)
    ax.set_xticks(FIG7_SNR)
    if metric.endswith('fraction') or metric.startswith('success'):
        ax.set_ylim(-.03,1.03)
    ax.grid(alpha=.3);ax.legend(fontsize=8,ncol=2)
    fig.tight_layout();fig.savefig(dst,dpi=180);plt.show();plt.close(fig)


def save_meta(outdir, source_paths, raw_metas, model_path=None, oracle_path=None):
    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)
    source=[]
    for p,meta in zip(source_paths,raw_metas):
        p=Path(p)
        source.append(dict(path=str(p),sha256=file_sha256(p),source_metadata=meta))
    ck={}
    for key,p in (('jscc_best',model_path),('oracle_best',oracle_path)):
        if p and Path(p).exists(): ck[key]=dict(path=str(p),sha256=file_sha256(p))
    # Only JSCC's SHA was stored by the first notebook. Warn if it changed.
    for v in source:
        md=v['source_metadata']
        if md and md.get('model_sha256') and 'jscc_best' in ck:
            if md['model_sha256']!=ck['jscc_best']['sha256']:
                raise ValueError(f'{v["path"]}: Source JSCC SHA differs from present checkpoint!')
    m={'sources':source,'present_checkpoints':ck,
       'legacy_oracle_sha_warning':'An original CSV cannot be tied to an OraNet checkpoint if its sidecar lacks oracle_sha256',
       'dataset':'CIFAR10 validation vs CIFAR100 test','channel':'AWGN',
       'paper_doi':'10.1109/TWC.2023.3234408',
       'strictness':'No training or high-resolution data. Original CSV evaluation noise repeats may equal one.'}
    (outdir/'audit_manifest.json').write_text(json.dumps(m,indent=2,ensure_ascii=False,default=str))
    return m


## 2. 결과 CSV·체크포인트 무결성 확인

검증할 사항: 실제 이미지 수, 완전한 SNR×CR grid, 물리 CR (`active_channels / 96`), JSCC SHA. 기존 노트북의 CSV 메타데이터에는 **OraNet SHA가 없으므로** 예측값의 가중치 출처를 암호학적으로 단정할 수 없습니다. 바로 다음 셀에 checkpoint spot-check가 있습니다.

In [ ]:
val, val_meta = read_grid(CIFAR_VAL_CSV, 'CIFAR10_validation', expected_n=EXPECTED_IMAGES)
test, test_meta = read_grid(CIFAR_TEST_CSV, 'CIFAR100_test', expected_n=EXPECTED_IMAGES)

vcube = to_cube(val)
tcube = to_cube(test)
print('Validation cube:',vcube.actual.shape, 'Test cube:',tcube.actual.shape)
print('SNR (paper):',tcube.snrs)
print('Active channels:',tcube.channels)
print('Effective physical rates:',tcube.rates)
assert np.array_equal(vcube.channels,tcube.channels)
assert np.allclose(vcube.snrs,tcube.snrs)

provenance = save_meta(OUT,[CIFAR_VAL_CSV,CIFAR_TEST_CSV],[val_meta,test_meta],JSCC_BEST,ORACLE_BEST)
print('JSCC/checkpoint metadata validated; provenance saved.')
if ORACLE_BEST.exists(): print('Current OraNet SHA256:',provenance['present_checkpoints']['oracle_best']['sha256'])

### 2-1. 기존 OraNet 체크포인트와 CSV의 예측값 일치 검사

기존 평가 CSV의 메타데이터는 **JSCC 체크포인트만 인증**합니다. 아래 선택적 셀은 실제 CIFAR100 이미지를 현재 체크포인트의 encoder/OraNet에 넣고, CSV의 PSNR 예측값과 비교합니다. 학습 없이 소량의 이미지에 대해서만 실행되며, 기본값은 ON입니다.

이 스팟 체크는 임의로 선택한 수 개 샘플의 일치만 입증합니다. 전체 800,000행의 일치를 입증하려면 신규 서명(두 모델 SHA 포함)으로 전체 평가를 재생성해야 합니다.

In [ ]:
def verify_prediction_fingerprint(test_df, jscc_path, oracle_path, cifar_root,
                                  tolerance_db=0.025, seed=2026):
    import subprocess
    import torch
    from torchvision.datasets import CIFAR100
    from torchvision.transforms import ToTensor
    from torch import nn

    if not Path(jscc_path).exists() or not Path(oracle_path).exists():
        raise FileNotFoundError('JSCC/OraNet best checkpoints required for fingerprint spot-check')
    repo = Path('/content/PADC_author_original')
    repo_url='https://github.com/wyzhang-ustb/Predictive-and-Adaptive-Deep-Coding-for-Wireless-Image-Transmission-in-Semantic-Communication.git'
    if not (repo/'models.py').is_file():
        subprocess.run(['git','clone','--depth','1',repo_url,str(repo)],check=True)
    # Pin to training-time GitHub SHA from manifest when possible.
    manifest = RESULT_DIR/'manifest.json'
    if manifest.exists():
        upstream = json.loads(manifest.read_text()).get('upstream_git_sha')
        if upstream:
            now=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
            if now!=upstream:
                subprocess.run(['git','-C',str(repo),'fetch','--depth','1','origin',upstream],check=True)
                subprocess.run(['git','-C',str(repo),'checkout','--detach',upstream],check=True)
    if str(repo) not in sys.path: sys.path.insert(0,str(repo))
    from models import ADJSCC_V, AF_block
    from OracleNet import OracleNet

    def attention_forward(self, x, snr):
        bs=x.size(0)
        pooled=x.mean(dim=(2,3))
        inp=torch.cat([pooled,snr.reshape(bs,1).to(x.device,dtype=x.dtype)],dim=1)
        weights=self.sigmoid(self.fc2(self.relu(self.fc1(inp))))
        return x*weights.reshape(bs,-1,1,1)
    AF_block.forward=attention_forward

    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    j=ADJSCC_V([48,8,8],5,256).to(device).eval()
    o=OracleNet(48).to(device).eval()
    # These checkpoint files were created by this notebook, not arbitrary external content.
    j.load_state_dict(torch.load(jscc_path,map_location=device,weights_only=False)['state_dict'])
    o.load_state_dict(torch.load(oracle_path,map_location=device,weights_only=False)['state_dict'])

    ds=CIFAR100(root=str(cifar_root),train=False,download=False)
    rng=np.random.default_rng(seed)
    ids=np.array([0,1,17,101,999,4001,7000,9999],dtype=int)
    assert len(ds)>=ids.max()+1
    x=torch.stack([ToTensor()(ds[int(k)][0]) for k in ids]).to(device)
    out_rows=[]
    with torch.inference_mode():
        for snr_db in [0.,12.,24.]:
            snr=torch.full((len(ids),1),snr_db,device=device)
            z=j.encoder(x,snr).reshape(len(ids),48,8,8)
            mu=z.float().mean((2,3));sd=z.float().std((2,3),unbiased=True)
            for nactive in [6,8,16,32]:
                nch=torch.full((len(ids),1),float(nactive),device=device)
                feat=torch.cat([mu,sd,snr.float(),nch],dim=1)
                predicted=o.relu(o.fc3(o.fc2(o.fc1(feat.float())))).reshape(-1).cpu().numpy()
                for i,k in enumerate(ids):
                    im=f'img_{k:05d}'
                    hits=test_df[(test_df.image_id==im)&(test_df.snr_db==snr_db)&(test_df.active_channels==nactive)]
                    if len(hits)!=1:raise ValueError(f'Cannot identify unique cached prediction {im},{snr_db},{nactive}')
                    cached=float(hits.predicted_psnr_db.iloc[0]);diff=float(predicted[i]-cached)
                    out_rows.append(dict(image_id=im,snr_db=snr_db,active_channels=nactive,
                                         current_checkpoint_pred_db=float(predicted[i]),cached_csv_pred_db=cached,
                                         difference_db=diff,abs_error_db=abs(diff)))
    audit=pd.DataFrame(out_rows)
    max_err=float(audit.abs_error_db.max())
    if max_err>float(tolerance_db):
        raise AssertionError(f'Cached predictions and CURRENT OraNet checkpoint differ by {max_err:.4f} dB! '
                             'Do NOT use these CSVs as new-checkpoint results. Re-evaluate to a NEW CSV path.')
    print(f'PASS: {len(audit)} matching cached predictions (max abs difference {max_err:.6f} dB)')
    return audit

if RUN_CHECKPOINT_SPOTCHECK:
    finger=verify_prediction_fingerprint(test,JSCC_BEST,ORACLE_BEST,CIFAR100_ROOT,
                                         SPOTCHECK_TOLERANCE_DB,SEED)
    finger.to_csv(OUT/'cached_oracle_checkpoint_fingerprint.csv',index=False)
    display(finger.groupby('snr_db',as_index=False)['abs_error_db'].agg(['mean','max']))
else:
    print('Checkpoint prediction fingerprint skipped. Cache-to-OraNet provenance remains unverified.')

## 3. Fig. 7(a) — CIFAR100에서 OraNet의 예측 MSE

공식 Fig. 7(a)의 세 물리 CR은 `R=1/16, 1/12, 1/6`입니다. 지표는 `mean((predicted_PSNR - actual_PSNR)**2)`이고, **평균 PSNR 두 개의 차이를 제곱한 값이 아닙니다**. 색상 밴드는 이미지 표본 평균 MSE의 근사 95% 신뢰구간이며, 채널 반복 횟수는 별도로 기록합니다.

In [ ]:
fig7 = fig7_metrics(test)
fig7.to_csv(OUT/'fig7a_cifar100_awgn_exact_mse.csv',index=False)
fig7_plot(fig7,OUT/'fig7a_cifar100_awgn_exact_mse.png')
print('Fig. 7(a) CIFAR100 MSE (dB²) — exact SNR and CR grid:')
display(fig7.pivot(index='snr_db',columns='physical_r',values='mse_db2').round(4))
print('Prediction bias by rate:')
display(fig7.groupby('physical_r')[['mse_db2','mae_db','signed_bias_db']].mean().round(4))
print('Caution: visual trend agreement is not equivalent to numeric curve equality; paper values are not digitized here.')

### 3-1. 논문 Fig. 7(a)와 대략적인 수치 비교 (선택적 참고)

아래 `PAPER_FIG7A_APPROX`는 **제공된 논문 PDF의 Fig. 7(a)를 픽셀 위치에서 읽은 근삿값**입니다. 저자가 공개한 원시 실험 데이터가 아닙니다. 따라서 차이값은 **그래프 수준의 sanity check**이지 엄밀한 동일성 인증이 아닙니다. 원본 그림의 작은 해상도와 마커 위치로 인해 수치 오차가 있을 수 있습니다.


In [ ]:
# Paper Fig.7(a) visually digitized approximate coordinates (NOT author raw data).
PAPER_FIG7A_APPROX = {
    6: [1.38, 0.90, 0.73, 0.66, 0.60, 0.60, 0.63, 0.67, 0.72],
    8: [1.22, 0.87, 0.66, 0.59, 0.51, 0.50, 0.55, 0.60, 0.63],
   16: [0.95, 0.75, 0.59, 0.53, 0.50, 0.53, 0.59, 0.69, 0.79],
}
paper_ref=pd.DataFrame([{'snr_db':float(s),'active_channels':int(ch),'paper_visual_approx_mse_db2':float(mse)}
        for ch,seq in PAPER_FIG7A_APPROX.items() for s,mse in zip(FIG7_SNR,seq)])
paper_ref.to_csv(OUT/'paper_fig7a_visual_approx_NOT_RAW.csv',index=False)
comparison=fig7.merge(paper_ref,on=['snr_db','active_channels'],validate='one_to_one')
comparison['difference_vs_paper_approx_db2']=comparison.mse_db2-comparison.paper_visual_approx_mse_db2
comparison.to_csv(OUT/'fig7a_comparison_to_paper_visual_approx.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5))
for ch,d in comparison.groupby('active_channels'):
    line=ax.plot(d.snr_db,d.mse_db2,'o-',lw=1.6,label=f'Our CSV R={ch/96:.4f}')[0]
    ax.plot(d.snr_db,d.paper_visual_approx_mse_db2,'x--',lw=1.2,color=line.get_color(),
            label=f'Paper figure approx R={ch/96:.4f}')
ax.set(xlabel='Channel SNR (dB)',ylabel='PSNR prediction MSE (dB$^2$)',
       title='CIFAR100 AWGN: reproduction vs paper Fig. 7(a) visual estimate')
ax.set_xticks(FIG7_SNR);ax.grid(alpha=.3);ax.legend(fontsize=8,ncol=2)
fig.tight_layout();fig.savefig(OUT/'fig7a_vs_paper_visual_approx.png',dpi=180);plt.show();plt.close(fig)
display(comparison.groupby('active_channels')['difference_vs_paper_approx_db2'].agg(
    mean_difference='mean',mean_absolute_difference=lambda e:e.abs().mean(),maximum_absolute_difference=lambda e:e.abs().max()).round(4))
print('CAUTION: reference MSE values are approximate visual readings, not publisher-provided numeric ground truth.')

## 4. Fig. 5-style — Data-level CR 결정

**논문 Fig. 5는 Data-level 최소 CR** 그래프입니다. 혼동을 막기 위해 세 개를 구분합니다.

1. **사후 논문형:** CIFAR100 테스트 데이터의 평균 PSNR을 알고 가장 작은 CR 선택 (동일 테스트셋을 본 사후 기준).
2. **현실적 선택:** CIFAR10 validation 평균 PSNR에서 CR을 결정하고 CIFAR100 평균 PSNR로 평가 (학습-테스트 데이터 도메인 차이 존재).
3. **Instance-level:** 이미지 내용별 OraNet 예측으로 CR 결정. 이건 Fig. 5의 Data-level 곡선이 아니라 **별도의 PADC 평가**입니다.

두 Data-level 방식 모두 `max CR = 1/3`에서 목표 달성이 불가능하면 최대 CR을 반환하면서 `feasible=False`를 기록합니다. 이번 비교에는 **ADJSCC 4모델 데이터가 없음**을 명시합니다.

In [ ]:
fig5_data = data_level_summary(vcube,tcube)
fig5_data.to_csv(OUT/'fig5_style_data_level_cifar100.csv',index=False)
fig5_plot(fig5_data,OUT/'fig5_style_deepjsccv_data_level.png')
display(fig5_data.head(18).round(5))
print('Paper-style data-level mean feasibility (CIFAR100):',round(fig5_data.paper_posthoc_cifar100_feasible.mean(),4))
print('Deployable validation-lookup mean PSNR success on CIFAR100:',round(fig5_data.data_cifar10_satisfies_mean_test.mean(),4))
print('These are SNR x target aggregate decisions, NOT per-image reliability.')

## 5. Instance-level PADC — Genie, feasibility, constraint violations

이 부분은 **논문 P2(이미지별 최소 CR)** 실험의 검증입니다. 원논문 Fig. 5와 구분해서 그립니다. 모든 후보 CR은 기존 CSV에서 `effective_r`로 정렬합니다.

- `feasible`: **실제 PSNR로** 목표를 만족하는 후보가 8개 중 하나라도 있음.
- `success_given_feasible`: 위 feasible 조건에 한해서 **OraNet이 고른 CR의 실제 PSNR이 목표 이상인 비율**.
- `mean_excess_rate_if_success`: 실제 목표를 달성한 feasible 사례에서만 `R_PADC-R_Genie` 평균. 실패 사례까지 섞어 rate가 더 작아 보이게 만들지 않음.
- `calibrated`: CIFAR10 validation에서 나온 **경험적 95% 과대예측 마진**을 빼고 선택하는 대조 실험. CIFAR10 validation이 이미 모델 선택에 사용되었으므로 통계적 보장이 아닙니다.

In [ ]:
margins = val_margin(val)
print('CIFAR10 validation overprediction margins (dB):',margins)
RAW = instance_level_summary(tcube,margin=0.0,tag='raw')
CAL = instance_level_summary(tcube,margin=margins['overprediction_quantile_95'],tag='val_empirical_q95')
INSTANCE = pd.concat([RAW,CAL],ignore_index=True)
INSTANCE.to_csv(OUT/'instance_level_feasible_only_by_snr_target.csv',index=False)
AGG=aggregate_instance(INSTANCE)
AGG.to_csv(OUT/'instance_level_weighted_summary.csv',index=False)
display(AGG.round(5))
print('Only feasible cases can be used to judge a PSNR-target guarantee:')
display(INSTANCE[(INSTANCE.target_psnr_db.isin([24.,28.,32.,34.]))&(INSTANCE.snr_db.isin([0.,12.,24.]))][
    ['variant','snr_db','target_psnr_db','feasible_fraction','success_given_feasible','mean_padc_rate_feasible',
     'mean_genie_rate_feasible','mean_excess_rate_if_success']].round(5))
instance_plot(INSTANCE,OUT/'instance_feasible_only_success.png',
              'success_given_feasible','CIFAR100 / AWGN: success conditional on feasible target')
instance_plot(INSTANCE,OUT/'instance_mean_selected_cr.png',
              'mean_padc_rate_all','CIFAR100 / AWGN: mean selected physical CR')

## 6. 정책 실패·비단조성 검사

PADC 논문에서는 PSNR이 CR에 대해 일반적으로 증가한다고 설명합니다. 그러나 단발 AWGN 관측과 재구성 네트워크에서는 실제로 일부 비단조 오차가 생길 수 있으므로 **exhaustive grid search를 사용하고 이분 탐색을 강제하지 않습니다**.

In [ ]:
mono=monotonicity_summary(tcube)
mono.to_csv(OUT/'rate_monotonicity_diagnostics.csv',index=False)
display(mono.round(5))
error=test.copy()
error['signed_error_db']=error.predicted_psnr_db-error.psnr_db
error['squared_error_db2']=error.signed_error_db**2
bycell=error[error.snr_db.isin(FIG7_SNR)].groupby(['snr_db','active_channels'],as_index=False).agg(
    prediction_bias_db=('signed_error_db','mean'),prediction_mse_db2=('squared_error_db2','mean'))
bycell.to_csv(OUT/'prediction_bias_by_snr_cr.csv',index=False)
print('Saved detailed bias and monotonicity diagnostics.')

## 7. 재현 보고서 및 ZIP 저장

위의 도표와 CSV는 `MyDrive/PADC_reproduction/paper/awgn/results/cifar_paper_fig7_fig5_reaudit/`에 별도 저장합니다. 기존 체크포인트 및 CSV는 **수정하지 않습니다**.

In [ ]:
import shutil
report={
    'article':'Zhang et al., IEEE TWC 2023; DOI 10.1109/TWC.2023.3234408',
    'channel':'AWGN','dataset':'CIFAR100 10000 test images',
    'fig7_snr_db':FIG7_SNR.tolist(),'fig7_nominal_physical_rates':['1/16','1/12','1/6'],
    'fig7_average_mse_db2':{str(x):float(g.mse_db2.mean()) for x,g in fig7.groupby('active_channels')},
    'data_level_paper_style_feasible_fraction':float(fig5_data.paper_posthoc_cifar100_feasible.mean()),
    'data_level_deployable_success_fraction':float(fig5_data.data_cifar10_satisfies_mean_test.mean()),
    'ora_prediction_overprediction_q95_val_db':float(margins['overprediction_quantile_95']),
    'instance_feasible_only':AGG.to_dict(orient='records'),
    'spotcheck_run':RUN_CHECKPOINT_SPOTCHECK,
    'caveats':['Original Fig5 also compares against 4 separately trained ADJSCC models: not reproduced here',
               'Original old-grid metadata does not include an OraNet SHA; spot-check validates only samples',
               'CIFAR10 test/validation was used for training-time model selection and is not a fresh calibration holdout',
               'Single channel realization per test/SNR/CR in the existing CIFAR100 paper-profile grid',
               'Post-hoc CIFAR100 lookup is not a deployable data-level strategy and uses test distribution knowledge',
               'No training was executed in this notebook']
}
(OUT/'CIFAR_PADC_STRICT_AUDIT_REPORT.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
print('Audit reports and figures:')
for p in sorted(OUT.glob('*')):print(' ',p.name)
archive=shutil.make_archive(str(OUT.parent/'PADC_CIFAR_strict_fig7_fig5_audit'), 'zip',root_dir=OUT)
print('ZIP:',archive)
print('Complete: existing models preserved, no retraining, no ADJSCC baseline invented.')